# CatDiffusion: an educational pixel-space DDPM

This notebook builds an unconditional denoising diffusion probabilistic model (DDPM) for cat images. It is **not Stable Diffusion**: it has no text conditioning, CLIP, latent autoencoder, classifier-free guidance, or pretrained diffusion pipeline.

We will implement and use three equations:

$$x_t=\sqrt{\bar\alpha_t}x_0+\sqrt{1-\bar\alpha_t}\epsilon,$$
$$\hat\epsilon=\epsilon_\theta(x_t,t),\qquad \mathcal L=\operatorname{MSE}(\epsilon,\hat\epsilon),$$
$$x_{t-1}=\mu_\theta(x_t,t)+\sqrt{\tilde\beta_t}z.$$

The notebook is deliberately honest about compute: a quick demo checks the pipeline, while useful image quality generally needs longer GPU training.

In [ ]:
from pathlib import Path
import sys, platform, torch
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('GPU memory (GiB):', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Selected device:', DEVICE)

## Install the project

For a published repository, replace the one placeholder below. In a local checkout, run the fallback install instead. Do not repeatedly download the dataset if it already exists.

In [ ]:
# Published Colab setup: replace <GITHUB_USERNAME>/<REPOSITORY> after publishing.
GITHUB_REPOSITORY = '<GITHUB_USERNAME>/<REPOSITORY>'
PROJECT = Path.cwd()
if not (PROJECT / 'catdiffusion').exists():
    if '<GITHUB_USERNAME>' in GITHUB_REPOSITORY:
        raise RuntimeError('Set GITHUB_REPOSITORY to the published repository URL, or run this notebook from the project root.')
    !git clone https://github.com/{GITHUB_REPOSITORY}.git catdiffusion-project
    PROJECT = Path('catdiffusion-project').resolve()
!python -m pip install -q -e {PROJECT}
sys.path.insert(0, str(PROJECT))

In [ ]:
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision.utils import make_grid
from catdiffusion.dataset import CatOnlyDataset
from catdiffusion.diffusion import DDPM
from catdiffusion.model import UNet
from catdiffusion.training import train_step

def display_grid(images, title):
    grid = make_grid(images[:16].cpu(), nrow=4).permute(1, 2, 0).add(1).div(2).clamp(0, 1)
    plt.figure(figsize=(8, 4)); plt.imshow(grid); plt.axis('off'); plt.title(title); plt.show()

IMAGE_SIZE = 64 if DEVICE == 'cuda' else 32
QUICK_DEMO = True  # Set False for a longer experiment.
dataset = CatOnlyDataset(root=PROJECT / 'data', split='trainval', image_size=IMAGE_SIZE,
                         augment=True, download=True, max_images=64 if QUICK_DEMO else None)
loader = DataLoader(dataset, batch_size=8 if QUICK_DEMO else 32, shuffle=True, num_workers=0)
batch = next(iter(loader))
display_grid(batch, 'Real cat training images')

## Forward diffusion and schedule

The schedule uses zero-based Python indices `i=0,...,T-1`, where `i` represents mathematical timestep `t=i+1`. The cosine schedule is useful for short educational runs because its terminal cumulative alpha is close to zero. The direct `q_sample` call below uses actual data, not an illustrative fabricated image.

In [ ]:
diffusion = DDPM(steps=100 if QUICK_DEMO else 300, schedule='cosine').to(DEVICE)
x0 = batch[:1].to(DEVICE)
levels = torch.linspace(0, diffusion.steps - 1, 6, dtype=torch.long, device=DEVICE)
noisy = [diffusion.q_sample(x0, t[None])[0].cpu() for t in levels]
fig, axes = plt.subplots(1, len(noisy), figsize=(15, 3))
for ax, image, t in zip(axes, noisy, levels):
    ax.imshow(make_grid(image, nrow=1).permute(1, 2, 0).add(1).div(2).clamp(0, 1)); ax.set_title(f'i={int(t)}'); ax.axis('off')
plt.tight_layout(); plt.show()
print('terminal alpha_bar:', float(diffusion.alpha_bars[-1]))

In [ ]:
plt.plot(diffusion.betas.cpu(), label='beta')
plt.plot(diffusion.alpha_bars.cpu(), label='cumulative alpha')
plt.xlabel('zero-based diffusion index'); plt.legend(); plt.title('Cosine noise schedule'); plt.show()

## Timestep-conditioned U-Net and training

The model predicts the Gaussian noise added by `q_sample`. `F.mse_loss` averages over tensor elements, which is the loss normalization used here. The quick demo is a smoke test, not a quality claim.

In [ ]:
model = UNet(in_channels=3, out_channels=3, base_channels=16 if DEVICE == 'cpu' else 32).to(DEVICE)
print('parameters:', sum(p.numel() for p in model.parameters()))
print('input/output:', tuple(batch.shape), tuple(model(batch[:1].to(DEVICE), torch.zeros(1, dtype=torch.long, device=DEVICE)).shape))
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)
losses = []
steps = 2 if QUICK_DEMO else 1000
iterator = iter(loader)
for step in range(steps):
    try: x, _ = next(iterator)
    except StopIteration: iterator = iter(loader); x, _ = next(iterator)
    loss = train_step(model, diffusion, x.to(DEVICE), optimizer, DEVICE)
    losses.append(float(loss));
print('last loss:', losses[-1])
plt.plot(losses); plt.xlabel('step'); plt.ylabel('MSE loss'); plt.title('Measured training loss'); plt.show()

In [ ]:
checkpoint = PROJECT / 'outputs' / 'notebook_checkpoint.pt'
checkpoint.parent.mkdir(exist_ok=True)
torch.save({'model': model.state_dict(), 'model_config': {'in_channels': 3, 'out_channels': 3, 'base_channels': 16 if DEVICE == 'cpu' else 32, 'image_size': IMAGE_SIZE}, 'diffusion_config': {'steps': diffusion.steps, 'schedule': 'cosine'}}, checkpoint)
with torch.inference_mode():
    samples, trajectory = diffusion.sample(model, (8, 3, IMAGE_SIZE, IMAGE_SIZE), DEVICE, seed=42, capture_steps=[0, diffusion.steps // 2, diffusion.steps - 1])
display_grid(samples.cpu(), 'Generated samples (quick demo)')

## Optional checkpoint loading and experiments

No pretrained checkpoint is bundled. To load one, provide a real, trusted checkpoint path produced by `scripts/train.py` and reconstruct the model/schedule from its metadata. Google Drive mounting is intentionally not automatic: if you need it in Colab, run `from google.colab import drive; drive.mount('/content/drive')` yourself and choose an explicit output path.

Suggested experiments: compare cosine and linear schedules; vary image size and `base_channels`; compare 100 and 300 reverse steps; inspect the effect of the random seed; and train longer while saving `training_loss.png` and `reverse_diffusion.png`. Keep generated figures tied to actual runs.